In [ ]:
from bbresults import BackboneScenarioAnalysis
from bbresults import BackboneResult
from functools import partial
import numpy as np
from gams import GamsWorkspace
import pandas as pd
import os
from pathlib import Path
import re
import subprocess as sp
from io import StringIO
from io import StringIO
from matplotlib.pyplot import get_cmap
from matplotlib.colors import to_hex
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pickle as pickle
import seaborn as sns
import plotly.io as pio   
pio.kaleido.scope.mathjax = None

In [ ]:
###Figure 6

In [ ]:
#market outcomes stored in script "1_ExAnte"
file = open("C:/Users/silke/WORK/3_Model/scripts/scripts/CfDs_2/pickle/moments",'rb')
df_mo= pickle.load(file)

In [ ]:
#pick bidding zones
nodes=["FR elec"]
df_mo=df_mo[df_mo['node'].apply(lambda x: 1 if any(i in x for i in nodes) else 0)==1]
#only new units
df_mo=df_mo[df_mo['unit'].apply(lambda x: 1 if any(i in x for i in ["New"]) else 0)==1]

In [ ]:
#load one scenario result from your path
res=BackboneResult("C:/Users/silke/WORK/3_Model/Results/20251410_CfD_Paper_2/Base_2019_Base_2019.gdx")

In [5]:
#add demand response price as max price
p_gnu_io=res.param_as_df("p_gnu_io")
unitUnittype=res.set_as_df("unitUnittype")
p_gnu_io=p_gnu_io.merge(unitUnittype,on="unit")
p_gnu_io["vomCosts"]=p_gnu_io[p_gnu_io['param_gnu']=='vomCosts']["value"]
dsr=p_gnu_io[p_gnu_io["unittype"]=="dsr"]
dsr=dsr[["node","vomCosts"]]
dsr=dsr.rename(columns={"vomCosts": "maxprice"})
maxprice = dsr['maxprice'].max()
df_mo.loc[
    (df_mo["price"] > maxprice) 
    #& (df_mo["price"] < 3999)
    ,
    "price"
] = maxprice

In [ ]:
#load all results from your path
res=BackboneScenarioAnalysis("C:/Users/silke/WORK/3_Model/Results/20251410_CfD_Paper_2")

In [ ]:
#retrieve capacities
p_gnu_io=pd.DataFrame()
for s, r in zip(res.scenarios, res.bb_results):     
    p_gnu=r.param_as_df("p_gnu_io")
    p_gnu=p_gnu[p_gnu['input_output']=='output']
    p_gnu["capacity"]=p_gnu[p_gnu['param_gnu']=='capacity']["value"]
    p_gnu["fomCosts"]=p_gnu[p_gnu['param_gnu']=='fomCosts']["value"]
    p_gnu["vomCosts"]=p_gnu[p_gnu['param_gnu']=='vomCosts']["value"]
    p_gnu["invCosts"]=p_gnu[p_gnu['param_gnu']=='invCosts']["value"]
    p_gnu["annuityFactor"]=p_gnu[p_gnu['param_gnu']=='annuityFactor']["value"]
    p_gnu=p_gnu.groupby(['node','grid','unit']).sum().reset_index()
    p_gnu=p_gnu[["node","grid","unit","vomCosts","fomCosts","invCosts","annuityFactor","capacity"]]
    p_gnu["scen"]=str(s)
    p_gnu_io=pd.concat([p_gnu_io,p_gnu])

In [ ]:
#w, f, p and their products
df_Q=df_mo.drop_duplicates(["unit","node","scen"]).drop(columns=["t","price","gen_t"])
df_Q=df_Q.merge(p_gnu_io,on=["node","unit","scen"])
df_Q_I=df_Q.groupby(["node","scen"])["capacity"].sum().reset_index().rename(columns={'capacity': 'capacity_node'})
df_Q=df_Q.merge(df_Q_I,on=["node","scen"])
df_Q["Q_Share"]=df_Q["capacity"]/df_Q["capacity_node"]
df_Q=df_Q.drop_duplicates(["unit","node","scen"])
df_Q=df_Q[["scen","unit","node","Q_Share","capacity","capacity_node"]]

#sum f and fp over T multiply with w
df_p=df_mo[["unit","node","scen","t","price","gen_t"]]
df_p=df_mo.merge(df_Q,on=["unit","node","scen"])
df_p["cf"]=(df_p["gen_t"]/df_p["capacity"])
df_p["fp"]=(df_p["price"]*df_p["cf"])
df_p["wfp"]=(df_p["fp"]*df_p["Q_Share"])
df_p=df_p[["unit","node","scen","fp","t","cf","wfp","price","gen_t"]]

In [ ]:
df_p["rev"]=df_p["price"]*df_p["gen_t"]
df_p["fp"]=df_p["price"]*df_p["cf"]
df_p=df_p.groupby(["unit","node","scen"]).sum().reset_index().rename(columns={'gen_t':'gen'})
df_p=df_p.merge(df_Q,on=["unit","node","scen"])
df_p["wfp"]=df_p["fp"]*df_p["Q_Share"]
#sum sum(f), sum(fp) and #sum(wfp) over i \in I in a node
df_node_rev=df_p.groupby(["node","scen"])[["cf","fp","wfp"]].sum().reset_index().rename(columns={"cf":"cf_node","fp":"fp_node","wfp":"wfp_node"})
df_p=df_p.merge(df_node_rev,on=["node","scen"])

#payment components
df_p=df_p.merge(p_gnu_io,on=["node","unit","scen","capacity"])
df_p["lcoe"]=df_p["vomCosts"]+(df_p["fomCosts"]+df_p["invCosts"]*df_p["annuityFactor"])/df_p["cf"]
df_p["v"]=df_p["fp_node"]/df_p["cf_node"]
df_p["v_i"]=df_p["fp"]/df_p["cf"]

df_p["ann_cost_mw"]=(df_p["vomCosts"]*df_p["cf"]+df_p["fomCosts"]+df_p["invCosts"]*df_p["annuityFactor"])
df_p["r"]=df_p["wfp_node"]
df_p["r_i"]=df_p["fp"]

df_p["node"]=df_p["node"].map(lambda x: str(x)[:2])
df_p["Profile"]=df_p["unit"].map(lambda x: str(x)[16:])
df_p["Profile"] = df_p["Profile"].str.replace(r'\bNew 1\b', 'High FLH', regex=True)
df_p["Profile"] = df_p["Profile"].str.replace(r'\bNew 2\b', 'High MV', regex=True)


In [ ]:
#load strike prices stored in script "1_ExAnte"
file = open("C:/Users/silke/WORK/3_Model/scripts/scripts/CfDs_2/pickle/strikeprice",'rb')
df_sp= pickle.load(file)

In [ ]:
#CfD payments
df_sp["node"]=df_sp["node"].map(lambda x: str(x)[:2])
df_payments=df_sp.merge(df_p,on=["node","unit"],how='left')
df_payments["subsidy"]=0
df_payments=df_payments[["scen","rev","unit","node","lcoe","v_i","v","gen","capacity","r_i","r","Profile","type","ann_cost_mw","strikeprice"]]

df_2way=df_payments[df_payments["type"]=="2way CfD"]
df_2way["subsidy"]=(df_2way["strikeprice"]-df_2way["v"])*df_2way["gen"]

df_fin=df_payments[df_payments["type"]=="Financial CfD"]
df_fin["subsidy"]=(df_fin["strikeprice"]-df_fin["r"])*df_fin["capacity"]

df_basic=df_payments[df_payments["type"]=="Basic CfD"]
df_basic["subsidy"]=df_basic["strikeprice"]*df_basic["gen"]-df_basic["rev"]

df_none=df_payments[df_payments["type"]=="Basic CfD"]
df_none["type"]="No CfD"
df_none["subsidy"]=0

In [16]:
#rate of cost recovery and cfd payments relative to market revenues
df_payments=pd.concat([df_basic,df_2way,df_fin,df_none])
df_payments["ann_cost"]=df_payments["ann_cost_mw"]*df_payments["capacity"]
df_payments["cost_rec"]=(df_payments["rev"]+df_payments["subsidy"])/df_payments["ann_cost"]*100

In [17]:
df_payments["mark_rec"]=df_payments["rev"]/df_payments["ann_cost"]*100
df_payments["subsidy_rec"]=df_payments["subsidy"]/df_payments["ann_cost"]*100
df_payments["receive"]=np.where(df_payments["subsidy_rec"]>0,df_payments["subsidy_rec"]+df_payments["mark_rec"],0)
df_payments["pay"]=np.where(df_payments["subsidy_rec"]<0,df_payments["mark_rec"],0)
df_payments["cost_rec2"]=df_payments["mark_rec"]+np.where(df_payments["subsidy_rec"]<0,df_payments["subsidy_rec"],0)

In [19]:
df_plot=df_payments[['node','unit','pay','receive','scen','mark_rec','cost_rec2','type']]
df_plot["unit"]=df_plot["unit"].map(lambda x: str(x)[3:])
df_plot["unit"][df_plot["unit"]=="Wind Onshore New 1"]="High FLH"
df_plot["unit"][df_plot["unit"]=="Wind Onshore New 2"]="High MV"
df_plot=df_plot.sort_values("node")

C:\Users\silke\AppData\Local\Temp\ipykernel_10360\3790961990.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_plot["unit"]=df_plot["unit"].map(lambda x: str(x)[3:])
C:\Users\silke\AppData\Local\Temp\ipykernel_10360\3790961990.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value



In [ ]:
#choose only one node and selected scenarios
nodes=["FR"]
scens=["Base 2019 Base 2019","BasePV- 2019 H2Price- 2015","H2Price-PV- 2019 Base 2010"]
df_plot2=df_plot[df_plot['node'].apply(lambda x: 1 if any(i in x for i in nodes) else 0)==1]
df_plot2=df_plot2[df_plot2['scen'].apply(lambda x: 1 if any(i in x for i in scens) else 0)==1]
df_plot2["scen"][df_plot2["scen"]=="Base 2019 Base 2019"]="H2Price+PVCost-H2Price+2019"
df_plot2["scen"][df_plot2["scen"]=="BasePV- 2019 H2Price- 2015"]="H2Price+PVCost+H2Price-2015"
df_plot2["scen"][df_plot2["scen"]=="H2Price-PV- 2019 Base 2010"]="H2Price-PVCost+H2Price+2010"
df_plot2["sorter"]="c"
df_plot2["sorter"][df_plot2["type"]=="No CfD"]="a"
df_plot2["sorter"][df_plot2["type"]=="Basic CfD"]="b"
df_plot2=df_plot2.sort_values(["sorter","type"],ascending=True)

C:\Users\silke\AppData\Local\Temp\ipykernel_10360\3728088331.py:6: FutureWarning:

ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


C:\Users\silke\AppData\Local\Temp\ipykernel_10360\3728088331.py:7: FutureWarning:

ChainedAssignmentError: behaviour will change

,node,unit,pay,receive,scen,mark_rec,cost_rec2,type,sorter
171,FR,High FLH,0.0,0.0,H2Price-PVCost+H2Price+2010,173.657514,173.657514,No CfD,a
160,FR,High FLH,0.0,0.0,H2Price+PVCost+H2Price-2015,52.067057,52.067057,No CfD,a
146,FR,High FLH,0.0,0.0,H2Price+PVCost-H2Price+2019,120.351576,120.351576,No CfD,a
207,FR,High MV,0.0,0.0,H2Price-PVCost+H2Price+2010,179.805572,179.805572,No CfD,a
196,FR,High MV,0.0,0.0,H2Price+PVCost+H2Price-2015,43.317530,43.317530,No CfD,a


In [ ]:
###Figure 6
fig=go.Figure(layout=dict(autosize=True,width=1000,height=450,
            plot_bgcolor='rgba(0,0,0,0)', title=dict(
            text="<b>FR</b>",        # Bold text using HTML tags
            x=0.43,                   # Centers the title horizontally
            xanchor='center',        # Anchor at the center
            yanchor='top'            # Anchor vertically at the top
        ),
            font_size=14,
            font_family="Open Sans", font_color="black",legend=dict(y=0.5)))
co1=["firebrick", "lightcoral"]
co2=["darkblue", "royalblue"]
co3=["grey", "lightgrey"]
a=0.4
for u, c1, c2, c3 in zip(df_plot2.unit.unique(),co1,co2,co3):

        df=df_plot2[df_plot2.unit==str(u)] 
        fig.add_trace(go.Bar(x=[df.scen,df.type],width=a,
        y=df.pay,name="Payment to government",offsetgroup=str(u),legendgroup=str(u),legendgrouptitle_text=str(u)
       ,marker_color=c1,marker_pattern_shape="/"))

        fig.add_trace(go.Bar(x=[df.scen,df.type],width=a,
        y=df.receive,name="Payment from government",offsetgroup=str(u),legendgroup=str(u),legendgrouptitle_text=str(u),marker_color=c2))

        fig.add_trace(go.Bar(x=[df.scen,df.type],width=a,
        y=df.cost_rec2.tolist(),offsetgroup=str(u),
        marker_color=c3,showlegend=False))

        fig.add_trace(go.Bar(x=[df.scen,df.type],width=a, 
        y=df.mark_rec.tolist(),name="Market Revenues",offsetgroup=str(u),legendgroup=str(u),legendgrouptitle_text=str(u),
        marker_color=c3,opacity=0.6))
        
fig.add_hline(y=100,line_width=1,line_color="red",line_dash="dash" )
fig.update_yaxes(linecolor="black",dtick=20,title="% of costs",title_standoff = 0.5,ticks="outside",range=[0,180])
fig.update_xaxes(linecolor="black",tickangle=270)

fig.show()